# Pope Francis abroad: where he goes and what he says

Python for data science project, ENSAE, 2026. Authors: Quentin Brisemur, Arthur De Rouck, Adrien Vibert.

Between 2013 and 2024, Pope Francis made 47 trips outside Italy. In each country, he gave several speeches to very different audiences: heads of state, bishops, young people, refugees, leaders of other religions. This report asks two questions. Where does the pope go? And once there, does he adapt what he says to the country and to the audience?

The report has three parts.

1. Where does the pope go? We compare the countries visited by John Paul II, Benedict XVI and Francis with their religious composition and their level of development.
2. What does he say? We analyse the 342 speeches that Francis gave during his trips: who he speaks to, which themes he raises in each country and on each continent, and whether his speeches depend more on the place or on the audience.
3. Are the differences robust? We check the gaps found in part 2 with a bootstrap and a negative binomial regression.

All the analyses of the speeches use the same corpus, built by the scripts of the `acquisition` folder.

The charts are interactive: hover over them to read the values.

The trips of the pope are both religious and diplomatic events. He meets heads of state, speaks before parliaments and international organisations, and his words are reported all over the world. Where he goes and what he says there tell us about the priorities of the Vatican. Before looking at the data, we expected Francis to favour poor and peripheral countries, in line with his call to go to "the peripheries". We also expected him to adapt his speeches strongly to each country, for example by talking about peace in countries at war and about migrants in countries of departure or arrival.

In [1]:
import sys
import textwrap
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "outils"))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

import analyse_corpus as ac
import english as en
import graphiques as g
import robustness as rb
from stockage import fichier

g.style()
pd.set_option("display.max_colwidth", 70)

## Part 1. Where does the pope go?

Following Pope Leo's visit to France in the fall of 2026 and given the media coverage it received, we decided to investigate the Vatican's diplomacy and to determine whether it pursues a geopolitical goal. To do this, we first analysed the countries visited by John Paul II, Benedict XVI and Francis. We cross the list of countries visited by these popes with the religious composition of these countries and their level of development.

Questions:

1. Who do the popes speak to? Are the visited countries mostly Catholic, or at least more Catholic than the world average?
2. Growth or decline? Do the popes go where Catholicism is growing or where it is declining?
3. Development: do they rather go to rich countries or to poor countries?

In [2]:
POPES = ["John Paul II", "Benedict XVI", "Francis"]
# Start and end of each pontificate
PONTIFICATES = {"John Paul II": (1978, 2005), "Benedict XVI": (2005, 2013), "Francis": (2013, 2025)}
POPE_COLORS = {"Francis": g.COULEURS[0], "John Paul II": g.COULEURS[1], "Benedict XVI": g.COULEURS[2]}

# Adrien's files are in his public folder of the SSP Cloud storage
ADRIEN = "avibert/diffusion"
FILE_TRIPS = "df_trips_countries.parquet"
FILE_RCS = "Religious Characteristics of States Dataset Project - Demographics v. 2.0 (RCS-Dem 2.0), COUNTRIES ONLY.SAV"
FILE_GDP = "API_NY.GDP.PCAP.KD_DS2_en_csv_v2_71952.csv"
FILE_COUNTRIES = "Metadata_Country_API_NY.GDP.PCAP.KD_DS2_en_csv_v2_71952.csv"

### The trips data: web scraping the Vatican website

As there isn't any source listing the popes' visits, we resorted to web scraping and got the list of the trips from the Vatican website directly. We complied with the document "robots.txt" and implemented a two seconds lap between two queries. The result has then been cleaned and saved on the S3 storage system. The scripts used are `scraping_no_speeches.py` and `cleaning_no_speeches.py` in the `acquisition` folder (the version with spaCy in `archives_cleaning_spacy.py` was not efficient enough and AI has been used to resolve the issues in the current script). For the analysis, we directly use the files from the S3 storage. The notebook copies them into `data/` the first time it runs.

In [3]:
trips = pd.read_parquet(fichier(FILE_TRIPS, ADRIEN))
trips = trips.rename(columns={"Pope": "pope", "Year": "year", "Place": "trip",
                              "Country": "country", "ISO3": "iso3"})
trips["pope"] = trips["pope"].replace({"John-Paul II": "John Paul II"})
trips["year"] = trips["year"].astype(int)

# The Vatican title of the 1985 trip names only the Netherlands, but John Paul II also went to
# Luxembourg and Belgium during this trip. We add the two missing countries.
netherlands_1985 = trips[(trips["pope"] == "John Paul II") & (trips["year"] == 1985) & (trips["iso3"] == "NLD")]
trips = pd.concat([trips, netherlands_1985.assign(country="Luxembourg", iso3="LUX"),
                   netherlands_1985.assign(country="Belgium", iso3="BEL")], ignore_index=True)

# Francis cancelled his trip to Dubai for the COP28 (December 2023) for health reasons: we remove it
trips = trips[~trips["trip"].str.contains("COP28")]
trips.head(3)

,pope,year,trip,country,iso3
0,John Paul II,1979,Ireland,Ireland,IRL
1,John Paul II,1979,Poland,Poland,POL
2,John Paul II,1979,"Dominican Republic, Mexico and Bahamas",Dominican Republic,DOM


### The religions: RCS-Dem 2.0

Our first step to enrich the dataset was to combine it with the estimated share of every religion in every country visited. We used the RCS-Dem 2.0 dataset from David Brown and Patrick James, uploaded on S3. The dataset is from 2019. Its structure is one row per country and per year (from 1900 to 2015). The columns ending in `PC` are shares of the population (in %), `TOTPP` is the total population. We keep the main religions and the detail of Christians. The rest of the population that is not included in any category (Chinese folk religions, African religions, syncretic Christians...) is put in an `other_religions` column, except the people whose religion is unknown (`UNKPC`). We decided to keep both Christians and their subdivisions to refine the analysis.

In [4]:
rcs_raw = pd.read_spss(fichier(FILE_RCS, ADRIEN))

# We give West Germany the ISO3 code of Germany
rcs_raw.loc[rcs_raw["ABBREV"] == "GFR", "ISO3"] = "DEU"

# We make the names of the columns more explicit
religions = pd.DataFrame({
    "iso3": rcs_raw["ISO3"].astype(str).str.strip(),
    "year": rcs_raw["YEAR"],
    "population": rcs_raw["TOTPP"],
    "christians": rcs_raw["CHRPC"],
    "muslims": rcs_raw["MUSPC"],
    "jews": rcs_raw["JEWPC"],
    "buddhists": rcs_raw["BUDPC"],
    "hindus": rcs_raw["HINPC"],
    "no_religion": rcs_raw["NREPC"],
    "catholics": rcs_raw["CATPC"],
    "protestants": rcs_raw["PRTPC"],
    "orthodox": rcs_raw["ORTPC"],
    "unknown": rcs_raw["UNKPC"],
})

# We create the "other religions" column
main_religions = ["christians", "muslims", "jews", "buddhists", "hindus", "no_religion"]
religions["other_religions"] = 100 - religions[main_religions].sum(axis=1) - religions["unknown"].fillna(0)
# Small negative values come from rounding in the data: we put them to 0
religions["other_religions"] = religions["other_religions"].clip(lower=0)
# If a country has no data on religions, the other religions are unknown too
religions.loc[religions["christians"].isna(), "other_religions"] = np.nan
# Some old states have no ISO3 code: we remove them
religions = religions[religions["iso3"] != ""]
religions = religions.dropna(subset=["year"])
religions["year"] = religions["year"].astype(int)
religions.head(3)

,iso3,year,population,christians,muslims,jews,buddhists,hindus,no_religion,catholics,protestants,orthodox,unknown,other_religions
0,USA,1790,3975839.0,97.079384,NaN,0.037728,0.0,0.0,0.531636,0.880317,95.266586,0.0,2.351252,0.000000
1,USA,1791,4096517.0,96.854424,NaN,0.039131,0.0,0.0,0.542168,0.913239,94.972534,0.0,2.564276,0.000002
2,USA,1792,4221052.0,96.627235,NaN,0.040227,0.0,0.0,0.552208,0.946115,94.678505,0.0,2.780326,0.000003


### GDP per capita, region and income (World Bank)

The second step to enrich the data was to add information on the economic development of the visited countries. GDP per capita in constant 2015 dollars (indicator `NY.GDP.PCAP.KD`) has been retrieved from the World Bank, downloaded from https://data.worldbank.org/indicator/NY.GDP.PCAP.KD (CSV).

In [5]:
gdp_raw = pd.read_csv(fichier(FILE_GDP, ADRIEN), skiprows=4)   # the first 4 lines are a title, not the table

# The last column is empty (the lines end with a comma): we remove the empty columns
gdp_raw = gdp_raw.dropna(axis=1, how="all")

# The data is in a wide format, we convert it with melt in a long format
wb = gdp_raw.drop(columns=["Country Name", "Indicator Name", "Indicator Code"])
wb = wb.melt(id_vars="Country Code", var_name="year", value_name="gdp_per_capita")
wb = wb.rename(columns={"Country Code": "iso3"})
wb["year"] = wb["year"].astype(int)
wb.head()

,iso3,year,gdp_per_capita
0,ABW,1960,NaN
1,AFE,1960,1186.271645
2,AFG,1960,NaN
3,AFW,1960,1130.438388
4,AGO,1960,NaN


In [6]:
# Region and income group of each country (current classification)
wb_countries = pd.read_csv(fichier(FILE_COUNTRIES, ADRIEN))
wb_countries = wb_countries[["Country Code", "Region", "IncomeGroup"]]
wb_countries = wb_countries.rename(columns={"Country Code": "iso3", "Region": "region", "IncomeGroup": "income"})
# The aggregates (World, Euro area...) have no region: we remove them
wb_countries = wb_countries.dropna(subset=["region"])
wb_countries.head(3)

,iso3,region,income
0,ABW,Latin America & Caribbean,High income
2,AFG,Middle East & North Africa,Low income
4,AGO,Sub-Saharan Africa,Lower middle income


### Join the trips with the data of the closest year

RCS-Dem stops in 2015. For each visit, we take the closest observation (`merge_asof`). We keep the gap in years to control it later on and make sure we do not use irrelevant data in our analysis. For the visits after 2015 (Francis), we assume that the share did not change drastically between 2015 and 2024 and use the 2015 data.

In [7]:
religions = religions.rename(columns={"year": "year_religion"}).sort_values("year_religion")

visits = pd.merge_asof(
    trips.sort_values("year"),
    religions,
    left_on="year", right_on="year_religion", by="iso3",
    direction="nearest")
visits["gap_religion"] = visits["year_religion"] - visits["year"]

In [8]:
# GDP: closest year, at most 5 years of difference
wb_sorted = wb.rename(columns={"year": "year_wb"}).sort_values("year_wb")
visits = pd.merge_asof(visits.sort_values("year"), wb_sorted,
                       left_on="year", right_on="year_wb", by="iso3",
                       direction="nearest", tolerance=5)
visits = visits.merge(wb_countries, on="iso3", how="left")

# Majority religion: we only compare the main religions and the other religions
majority_columns = main_religions + ["other_religions"]
# We drop the empty lines to compute the majority for the countries that have data only
visits["majority"] = visits[majority_columns].dropna(how="all").idxmax(axis=1)

### Check the join

In [9]:
# For each pope: how many visits without data, and what is the average gap in years?
visits.groupby("pope").agg(
    nb_visits=("iso3", "size"),
    no_religion_data=("christians", lambda x: x.isna().sum()),
    no_catholic_data=("catholics", lambda x: x.isna().sum()),
    mean_gap_religion=("gap_religion", lambda x: x.abs().mean()),
).loc[POPES].round(1)

,nb_visits,no_religion_data,no_catholic_data,mean_gap_religion
pope,,,,
John Paul II,202,0,0,0.0
Benedict XVI,26,0,0,0.0
Francis,69,0,0,3.6


In [10]:
# Visited countries where the other religions are the majority
visits.loc[visits["majority"] == "other_religions", ["pope", "year", "country", "christians", "other_religions"]].round(1).head(3)

,pope,year,country,christians,other_religions
11,John Paul II,1980,Ivory Coast,25.8,38.6
14,John Paul II,1980,Burkina Faso,11.9,44.0
25,John Paul II,1982,Benin,27.5,50.4


All the instructions went well and we do not see issues in the joins.

### Overview of the trips

The first step of our analysis is to get an overall view of the trips. We aim to answer the question: were trips equally important for every pope, and was every continent equally important for them? By comparing the different popes, we might find in the discrepancies signs that visits pursue specific goals other than mere pastoral visits.

In [11]:
# One trip = one trip title for a pope and a year
trips["trip_id"] = trips["pope"] + " " + trips["year"].astype(str) + " " + trips["trip"]

overview = trips.groupby("pope").agg(
    nb_trips=("trip_id", "nunique"),
    nb_visits=("iso3", "size"),
    nb_countries=("iso3", "nunique"),
).loc[POPES]
overview["pontificate_length"] = [end - start for start, end in PONTIFICATES.values()]
overview["trips_per_year"] = (overview["nb_trips"] / overview["pontificate_length"]).round(1)
overview

,nb_trips,nb_visits,nb_countries,pontificate_length,trips_per_year
pope,,,,,
John Paul II,105,202,128,27,3.9
Benedict XVI,24,26,22,8,3.0
Francis,47,69,64,12,3.9


In [12]:
per_year = trips.groupby(["year", "pope"])["trip_id"].nunique().rename("trips").reset_index()

fig = px.bar(per_year, x="year", y="trips", color="pope",
             category_orders={"pope": POPES}, color_discrete_map=POPE_COLORS)
fig.update_traces(marker_line_color=g.FOND, marker_line_width=1,
                  hovertemplate="%{x}, %{fullData.name}: %{y} trip(s)<extra></extra>")
fig.update_xaxes(title=None)
fig.update_yaxes(title="Number of trips", showgrid=True)
g.titre(fig, "Trips outside Italy each year", "Click on a pope in the legend to hide him", hauteur=430)
fig.show()

John Paul II and Francis have been more active than Benedict XVI, with 4 trips per year on average for the first two against 3 for the latter. John Paul II took advantage of his youth since he was elected at only 58. Francis, who was almost the same age as Benedict XVI, shows a strong willingness to visit countries despite his age. He travelled as much as John Paul II at the same age, about 4 trips per year.

In [13]:
# The most visited countries
per_country = (trips.groupby(["country", "pope"])["trip_id"].nunique()
                    .unstack(fill_value=0).reindex(columns=POPES, fill_value=0))
top = per_country.loc[per_country.sum(axis=1).sort_values().index[-15:]]

fig = go.Figure([go.Bar(x=top[pope], y=top.index, name=pope, orientation="h", marker_color=POPE_COLORS[pope],
                        marker_line_color=g.FOND, marker_line_width=1,
                        hovertemplate="%{y}, " + pope + ": %{x} trip(s)<extra></extra>")
                 for pope in POPES])
fig.update_layout(barmode="stack")
fig.update_xaxes(title="Number of trips", showgrid=True)
g.titre(fig, "The 15 most visited countries", "Number of trips that include the country", hauteur=520)
fig.show()

In [14]:
# Does John Paul II account for all the trips to Poland?
mask = (trips["pope"] == "John Paul II") & (trips["country"] == "Poland")
mask.sum()

np.int64(9)

The countries that are the most visited are mostly European or American. This would imply, as these countries are mostly Christian, that the Vatican opts more often for countries which have or used to have a predominantly Christian population. We were surprised by Poland's figure, but the peculiarity disappears when we see that John Paul II, born in Poland, accounts for 9 trips of the 11 in total.

### Maps of the visited countries

By representing the data on maps, we can visually see if some regions have been left aside and compare the popes' choices more easily. The buttons above the map show one pope at a time.

In [15]:
visited = (trips.groupby(["iso3", "pope"]).size().unstack(fill_value=0)
                .reindex(columns=POPES, fill_value=0) > 0)
country = trips.groupby("iso3")["country"].first()
n_popes = visited.sum(axis=1)

fig = go.Figure()
fig.add_choropleth(locations=visited.index, z=n_popes, text=country[visited.index], zmin=0.5, zmax=3.5,
                   colorscale=[[0, g.BLEUS[2]], [1 / 3, g.BLEUS[2]], [1 / 3, g.BLEUS[6]],
                               [2 / 3, g.BLEUS[6]], [2 / 3, g.BLEUS[11]], [1, g.BLEUS[11]]],
                   colorbar=dict(title="Popes", tickvals=[1, 2, 3], thickness=12, len=0.5),
                   hovertemplate="%{text}: visited by %{z} of the three popes<extra></extra>")
for pope in POPES:
    places = visited.index[visited[pope]]
    fig.add_choropleth(locations=places, z=[1] * len(places), text=country[places], visible=False,
                       showscale=False, colorscale=[[0, POPE_COLORS[pope]], [1, POPE_COLORS[pope]]],
                       hovertemplate="%{text}<extra>" + pope + "</extra>")
fig.update_traces(marker_line_color="white", marker_line_width=0.5)

views = ["All three popes"] + POPES
buttons = [dict(label=view, method="update", args=[{"visible": [i == k for i in range(len(views))]}])
           for k, view in enumerate(views)]
fig.update_layout(updatemenus=[dict(type="buttons", direction="right", buttons=buttons, showactive=True,
                                    x=0, xanchor="left", y=1.01, yanchor="bottom")],
                  margin=dict(l=0, r=0, b=0))
fig.update_geos(projection_type="natural earth", showframe=False, showcoastlines=False, lataxis_range=[-58, 85],
                showcountries=True, countrycolor="white", landcolor="#ecebe6", bgcolor=g.FOND)
g.titre(fig, "Countries visited by John Paul II, Benedict XVI and Francis", "Hover over a country to see its name",
        hauteur=560, marge_haute=45)
fig.show()

The maps show large differences between popes. Together, they have visited most countries except Russia and China (the largest countries left unvisited), the Horn of Africa and Central Asia. John Paul II roamed almost all of Europe and America, and most African countries. Benedict XVI focused on Europe and key countries in America (US, Mexico and Brazil). The comparison between Francis and John Paul II is interesting. Both visited numerous countries in America, but Francis picked mostly Eastern European and Asian countries, when his predecessor visited all of Europe indifferently and Africa.

### Who do the popes speak to?

To compare the visited countries with the others, we build a table with one row per pope and per country of the world, and a column `visited` which is True if the pope went there. Each country is counted once, even if the pope went there several times.

We use the religious data of the middle year of the pontificate (2015 for Francis, because RCS-Dem stops in 2015).

In [16]:
rows = []
for pope, (start, end) in PONTIFICATES.items():
    middle_year = min((start + end) // 2, 2015)
    table = religions[religions["year_religion"] == middle_year].copy()
    table["pope"] = pope
    visited_iso3 = trips.loc[trips["pope"] == pope, "iso3"]
    table["visited"] = table["iso3"].isin(visited_iso3)
    rows.append(table)

countries = pd.concat(rows)
countries["majority"] = countries[majority_columns].dropna(how="all").idxmax(axis=1)

# Number of visited countries and number of countries for each pope
countries.groupby("pope")["visited"].agg(["sum", "size"]).loc[POPES]

,sum,size
pope,,
John Paul II,128,235
Benedict XVI,22,234
Francis,64,233


The table is consistent with the previous results. There is a difference in size because Czechoslovakia was still an entry in 1991 and the Dutch West Indies in 2009.

#### Share of Catholics in the visited countries

In [17]:
visited_countries = countries[countries["visited"]]
visited_countries.groupby("pope")["catholics"].describe().loc[POPES].round(1)

,count,mean,std,min,25%,50%,75%,max
pope,,,,,,,,
John Paul II,127.0,38.4,33.7,0.0,5.6,26.9,77.1,95.2
Benedict XVI,22.0,44.9,31.2,0.1,23.0,40.5,70.3,93.1
Francis,64.0,33.0,32.1,0.0,1.5,24.2,63.4,92.6


In [18]:
# Same thing for the countries that were not visited, to compare
countries[~countries["visited"]].groupby("pope")["catholics"].describe().loc[POPES].round(1)

,count,mean,std,min,25%,50%,75%,max
pope,,,,,,,,
John Paul II,107.0,20.0,28.6,0.0,0.5,5.4,27.5,97.4
Benedict XVI,212.0,26.3,29.7,0.0,1.3,11.3,50.7,98.7
Francis,169.0,25.3,28.5,0.0,2.1,11.3,44.5,99.4


The tables confirm the hypothesis made earlier that the popes were opting more often for Catholic countries than others. The mean share of Catholics in visited countries is significantly higher than in the other countries. One issue is that the figure alone does not account for religiously diverse countries, where Catholicism could be the predominant religion without representing a high share of the population. Hence the following analysis.

#### How many visited countries are Christian or Catholic?

- Christian: Christians are the largest religion of the country (`majority` column).
- Catholic: Catholics are the largest group of the country when Christians are split into Catholics, Protestants and Orthodox and compared with the other religions (the `christians` column is left aside).

In [19]:
visited_countries = countries[countries["visited"]].copy()
visited_countries["christian_majority"] = visited_countries["majority"] == "christians"

# Largest group when the Christians are split into Catholics, Protestants and Orthodox
detail_columns = ["catholics", "protestants", "orthodox"] + [c for c in majority_columns if c != "christians"]
largest_group = visited_countries[detail_columns].dropna(how="all").idxmax(axis=1)
visited_countries["catholic_majority"] = largest_group.reindex(visited_countries.index) == "catholics"

result = visited_countries.groupby("pope")[["christian_majority", "catholic_majority"]].sum()
result["nb_visited_countries"] = visited_countries.groupby("pope").size()
result["% christian_majority"] = (result["christian_majority"] / result["nb_visited_countries"] * 100).round(1)
result["% catholic_majority"] = (result["catholic_majority"] / result["nb_visited_countries"] * 100).round(1)
result.loc[POPES]

,christian_majority,catholic_majority,nb_visited_countries,% christian_majority,% catholic_majority
pope,,,,,
John Paul II,92,62,128,71.9,48.4
Benedict XVI,18,14,22,81.8,63.6
Francis,43,30,64,67.2,46.9


The table confirms that popes are more likely to visit countries with a Christian and in particular a Catholic majority. The large percentages for a Christian majority are not completely surprising, as Protestant countries such as the US or Germany also have an important Catholic population, albeit in minority.

#### Distribution of the share of Catholics

The means can hide very different situations. We compare the whole distribution of the share of Catholics in the visited and in the non-visited countries. Each country counts once, and the histograms are normalised so that both groups can be compared despite their different sizes.

In [20]:
fig = make_subplots(rows=1, cols=3, subplot_titles=POPES, shared_yaxes=True, horizontal_spacing=0.04)
for column, pope in enumerate(POPES, start=1):
    table = countries[countries["pope"] == pope]
    for status, color, is_visited in [("Not visited", g.GRIS, False), ("Visited", g.COULEURS[0], True)]:
        fig.add_histogram(x=table.loc[table["visited"] == is_visited, "catholics"].dropna(),
                          xbins=dict(start=0, end=100, size=5), histnorm="probability density",
                          name=status, legendgroup=status, showlegend=column == 1,
                          marker=dict(color=color, line=dict(color=g.FOND, width=1)), opacity=0.6,
                          hovertemplate=status + ", %{x} %<extra></extra>", row=1, col=column)
fig.update_layout(barmode="overlay", legend_y=1.09)
fig.update_xaxes(title="Share of Catholics (%)", range=[0, 100], showgrid=True)
fig.update_yaxes(title_text="Density", showgrid=True, col=1)
g.titre(fig, "Distribution of the share of Catholics, visited and non-visited countries",
        "Each country counts once, the two groups are normalised", hauteur=470, marge_haute=30)
fig.show()

The non-visited countries are concentrated near 0 %: most countries of the world have almost no Catholics. The three popes differ in how far they move away from this pattern.

- John Paul II: the visited countries are much less often at 0 % than the non-visited ones, and there is a clear bump between 75 % and 90 %. He went to almost every very Catholic country, but he also visited many countries with few Catholics, which is consistent with the number of countries he covered.
- Benedict XVI: the countries with almost no Catholics are rare among his visits. His visits are spread between 20 % and 90 %, which matches his focus on Europe and on large Catholic countries of America.
- Francis: the two distributions are very close. The visited countries are as often near 0 % as the non-visited ones (Asia, Middle East), and the bump around 80 % is smaller. The difference of means for Francis therefore relies on a limited number of very Catholic countries, and his trips are the least oriented towards Catholic countries of the three.

### How does the Catholic population evolve in visited countries?

To go further in the analysis, we thought that looking at the dynamics of the Catholic population at the time of the visit could also be interesting. The goal is to determine whether the Vatican has a defensive or an offensive diplomacy. Does it target countries where the share of Catholics increases, or where it decreases, to attempt to halt its decline?

For each visit, we compute the change of the share of Catholics in the visited country during the 5 years before the visit (in percentage points). We compare it with the change of the average share of Catholics of all countries during the same 5 years (each country counts for one, like above).

RCS-Dem stops in 2015: for the visits after 2015, we use the change between 2010 and 2015.

In [21]:
# Average share of Catholics of all countries, each year
mean_catholics = religions.groupby("year_religion")["catholics"].mean().reset_index(name="mean_catholics")

# The first visit is in 1979, so the first year we need is 1974
fig = px.line(mean_catholics[mean_catholics["year_religion"] >= 1974], x="year_religion", y="mean_catholics")
fig.update_traces(line=dict(color=g.COULEURS[0], width=2),
                  hovertemplate="%{x}: %{y:.1f} % on average<extra></extra>")
fig.update_xaxes(title="Year")
fig.update_yaxes(title="Share of Catholics (%)", showgrid=True)
g.titre(fig, "Average share of Catholics of all countries", "Each country counts for one", hauteur=400)
fig.show()

We see that the share of Catholics is declining in general. Consequently, we expect the pope to visit countries where the Catholic population is also receding. A different result would imply an active diplomacy targeting specific countries.

In [22]:
# The 5 years before the visit (the data stops in 2015)
visits["year_end"] = visits["year"].clip(upper=2015)
visits["year_start"] = visits["year_end"] - 5

# Share of Catholics in the visited country at the start and at the end
share_start = religions[["iso3", "year_religion", "catholics"]].rename(
    columns={"year_religion": "year_start", "catholics": "catholics_start"})
share_end = religions[["iso3", "year_religion", "catholics"]].rename(
    columns={"year_religion": "year_end", "catholics": "catholics_end"})
visits = visits.merge(share_start, on=["iso3", "year_start"], how="left")
visits = visits.merge(share_end, on=["iso3", "year_end"], how="left")

# Average share of Catholics of all countries at the start and at the end
mean_start = mean_catholics.rename(columns={"year_religion": "year_start", "mean_catholics": "mean_start"})
mean_end = mean_catholics.rename(columns={"year_religion": "year_end", "mean_catholics": "mean_end"})
visits = visits.merge(mean_start, on="year_start", how="left")
visits = visits.merge(mean_end, on="year_end", how="left")

# Changes in percentage points
visits["change_country"] = visits["catholics_end"] - visits["catholics_start"]
visits["change_mean"] = visits["mean_end"] - visits["mean_start"]
visits["change_vs_mean"] = visits["change_country"] - visits["change_mean"]

#### Descriptive statistics

In [23]:
changes = visits.dropna(subset=["change_country"]).copy()
changes.groupby("pope")["change_country"].describe().loc[POPES].round(2)

,count,mean,std,min,25%,50%,75%,max
pope,,,,,,,,
John Paul II,201.0,-0.62,2.13,-8.15,-1.42,-0.15,0.30,10.69
Benedict XVI,26.0,-1.23,2.09,-8.56,-1.75,-0.80,-0.08,2.23
Francis,69.0,-0.27,2.52,-7.78,-1.14,-0.03,0.29,10.56


In [24]:
# Average change in the visited countries and in all countries
changes.groupby("pope")[["change_country", "change_mean", "change_vs_mean"]].mean().loc[POPES].round(2)

,change_country,change_mean,change_vs_mean
pope,,,
John Paul II,-0.62,-0.42,-0.20
Benedict XVI,-1.23,-0.49,-0.74
Francis,-0.27,-0.57,0.31


In [25]:
# Share of visits where Catholicism was growing, and growing faster than on average
changes["growing"] = changes["change_country"] > 0
changes["faster_than_mean"] = changes["change_vs_mean"] > 0

(changes.groupby("pope")[["growing", "faster_than_mean"]].mean() * 100).loc[POPES].round(1)

,growing,faster_than_mean
pope,,
John Paul II,41.8,55.7
Benedict XVI,19.2,34.6
Francis,43.5,65.2


In [26]:
fig = px.box(changes, x="change_vs_mean", y="pope", color="pope", points="all", hover_name="country",
             custom_data=["year"], orientation="h",
             category_orders={"pope": POPES}, color_discrete_map=POPE_COLORS)
fig.update_traces(marker=dict(size=5, opacity=0.5), line_width=1.5, jitter=0.6, showlegend=False,
                  hovertemplate="%{hovertext} %{customdata[0]}: %{x:+.1f} points<extra></extra>")
fig.add_vline(x=0, line=dict(color=g.ENCRE_2, width=1, dash="dot"))
fig.update_xaxes(title="Percentage points", showgrid=True)
fig.update_yaxes(title=None)
g.titre(fig, "Change of the share of Catholics, compared with the average of all countries",
        "5 years before the visit, one point per visit", hauteur=460)
fig.show()

The data is very enlightening. We see a great difference between Francis and Benedict XVI. Benedict visited countries that were losing Catholics at a faster pace than the mean. On the opposite, the countries which were visited by Francis show an average change of almost 0 (-0.27 points), when the average of all countries lost 0.57 points, and two thirds of his visits were in countries where Catholicism did better than the average. This shows a tropism toward countries where Catholicism resists or expands. We can therefore argue that Francis had an "offensive diplomacy" whereas Benedict XVI was more defensive. John Paul II occupies the middle ground.

To complete the analysis, a further analysis of the countries themselves and of the evolution of their population in terms of religion would help. For example, we could look, for Pope Francis, whether the countries with few Catholics are the ones with the greatest evolution or not.

### Development: do the popes go to rich or poor countries?

Last analysis before getting to the speeches, we wanted to look at the visited countries' level of development. This would supplement the analysis of the targeted population in terms of religion with an economic perspective.

For each visit, we divide the GDP per capita of the visited country by the world GDP per capita of the same year (World Bank, code `WLD`). A ratio above 1 means that the visited country is richer than the world average. We use the median and not the mean, because a few very rich countries (Luxembourg, Switzerland...) pull the mean up.

In [27]:
# World GDP per capita each year (the World Bank gives it with the code "WLD")
world_gdp = wb[wb["iso3"] == "WLD"][["year", "gdp_per_capita"]]
world_gdp = world_gdp.rename(columns={"gdp_per_capita": "world_gdp_per_capita"})

visits = visits.merge(world_gdp, on="year", how="left")
visits["gdp_ratio"] = visits["gdp_per_capita"] / visits["world_gdp_per_capita"]

# Visits without GDP data
visits.loc[visits["gdp_per_capita"].isna(), ["pope", "year", "country"]]

,pope,year,country
1,John Paul II,1979,Poland
20,John Paul II,1981,Guam
30,John Paul II,1982,San Marino
39,John Paul II,1983,Poland
55,John Paul II,1985,Liechtenstein
82,John Paul II,1987,Poland
104,John Paul II,1989,Reunion
107,John Paul II,1990,Curacao
285,Francis,2023,South Sudan


Only 9 visits out of 297 have no GDP data. They are mostly the trips of John Paul II to Poland before 1990 (the World Bank series starts after the fall of the communist regime) and very small territories (Guam, San Marino, Liechtenstein...). For Francis, only South Sudan is missing. We drop these visits: it slightly reduces the sample of John Paul II but should not change the results.

In [28]:
gdp = visits.dropna(subset=["gdp_ratio"]).copy()
gdp["richer_than_world"] = gdp["gdp_ratio"] > 1

gdp_table = gdp.groupby("pope").agg(
    nb_visits=("gdp_ratio", "size"),
    median_gdp_visits=("gdp_per_capita", "median"),
    median_gdp_world=("world_gdp_per_capita", "median"),
    median_ratio=("gdp_ratio", "median"),
    pct_richer_than_world=("richer_than_world", "mean"),
).loc[POPES]
gdp_table["pct_richer_than_world"] = gdp_table["pct_richer_than_world"] * 100
gdp_table.round(2)

,nb_visits,median_gdp_visits,median_gdp_world,median_ratio,pct_richer_than_world
pope,,,,,
John Paul II,194,4779.23,6741.22,0.70,41.75
Benedict XVI,26,23130.60,9265.53,2.45,73.08
Francis,68,10828.16,10822.57,0.99,50.00


The world GDP per capita is not the same for the three popes because it grows over time: this is why we compare the ratios and not the GDP directly.

The three popes have very different profiles. Half of the visits of John Paul II were in countries at least 30 % poorer than the world average (median ratio of 0.70), and only 42 % of his visits were in countries richer than the world. On the opposite, Benedict XVI mostly went to rich countries: the median visited country was almost 2.5 times richer than the world and 73 % of his visits were above the world average. Francis is exactly in the middle, with a median ratio of 1 and half of his visits on each side.

This is consistent with the previous parts: Benedict XVI focused on Europe and on a few large American countries, while John Paul II covered a large part of Africa and Latin America. However, the GDP ratio counts every visit, so the countries visited several times (Poland, France, Spain...) weigh more. We therefore also look at the income groups and at the number of different countries visited.

#### Number of visits by income group

We use the World Bank income groups. Be careful: it is the current classification, not the one of the year of the visit.

In [29]:
# Number of visits in each income group (from the poorest to the richest)
income_order = ["Low income", "Lower middle income", "Upper middle income", "High income"]
by_income = pd.crosstab(visits["income"], visits["pope"]).loc[income_order, POPES]

fig = go.Figure([go.Bar(x=income_order, y=by_income[pope], name=pope, marker_color=POPE_COLORS[pope],
                        marker_line_color=g.FOND, marker_line_width=2,
                        hovertemplate="%{x}, " + pope + ": %{y} visits<extra></extra>") for pope in POPES])
fig.update_layout(barmode="group")
fig.update_yaxes(title="Number of visits", showgrid=True)
g.titre(fig, "Number of visits by income group of the visited country", hauteur=440)
fig.show()
by_income

pope,John Paul II,Benedict XVI,Francis
income,,,
Low income,16,0,6
Lower middle income,42,4,8
Upper middle income,48,4,22
High income,95,18,33


In [30]:
# Number of different countries visited in each income group (a country visited twice counts once)
countries_by_income = visits.groupby(["income", "pope"])["iso3"].nunique().unstack(fill_value=0).loc[income_order, POPES]

# Total number of countries in each income group, to compare
countries_by_income["nb_countries"] = wb_countries["income"].value_counts()

# Share of the countries of the group that each pope visited (in %)
for pope in POPES:
    countries_by_income["% " + pope] = (countries_by_income[pope] / countries_by_income["nb_countries"] * 100).round(1)
countries_by_income

pope,John Paul II,Benedict XVI,Francis,nb_countries,% John Paul II,% Benedict XVI,% Francis
income,,,,,,,
Low income,14,0,6,25,56.0,0.0,24.0
Lower middle income,31,4,8,47,66.0,8.5,17.0
Upper middle income,33,4,22,59,55.9,6.8,37.3
High income,49,14,28,86,57.0,16.3,32.6


In number of visits, the high income countries come first for the three popes (about half of the visits of John Paul II and Francis, more than two thirds for Benedict XVI). This seems to contradict the GDP ratio of John Paul II, but it comes from the classification: it is the current one, and many countries he visited in the 1980s and 1990s are now high income (Poland, South Korea, Chile, Uruguay, Croatia...). The GDP of the year of the visit is therefore a better indicator for him.

The share of the countries of each group that were visited is more enlightening.

- John Paul II visited between 56 % and 66 % of the countries of every group. He did not favour rich or poor countries: he simply went almost everywhere.
- Benedict XVI never went to a low income country and visited only a few lower and upper middle income countries. His trips are clearly oriented towards rich countries.
- Francis visited about a third of the upper middle and high income countries, but only 17 % of the lower middle income countries and 24 % of the low income ones. Despite his discourse on the "peripheries" and the "Church of the poor", his trips do not target the poorest countries more than the others. Among the six low income countries he visited, three were at war or had just come out of one (Central African Republic, South Sudan, Democratic Republic of the Congo), which suggests that peace mattered at least as much as poverty in his choices.

We should keep in mind that the high income group also contains many small territories (Caribbean islands, Gulf states, micro-states) that popes rarely visit, which lowers the share of visited countries in this group.

To complete the analysis, we could cross the income group with the share of Catholics, to see whether the popes go to poor countries only when they are Catholic, or whether poverty is a criterion on its own.

## Part 2. What does he say?

We now look at what Francis says during his trips. We collected all the speeches he gave abroad and compare them by continent, by audience and over time.

### The speeches

The texts come from vatican.va. Two scripts of the `acquisition` folder produce the data.

1. `telecharger_discours.py` reads the list of trips outside Italy and their dates. It then goes through the monthly index of the speeches and keeps those given during a trip. This gives 415 texts.
2. `preparer_corpus.py` separates the header of each text (trip, audience, place, date) from the speech itself. It removes the press conferences on the plane, which mostly contain the questions of journalists, and two speeches given in the Vatican on the morning of a departure. 342 speeches remain.

The files are stored on the SSP Cloud. The notebook reads them directly, without running the collection again, which takes about ten minutes.

In [31]:
df = en.translate(ac.charger())
df[["date", "voyage", "pays", "public", "n_mots", "titre"]].head()

,date,voyage,pays,public,n_mots,titre
0,2013-07-22,Rio de Janeiro 2013,Brazil,Welcome and farewell,990,Welcome Ceremony at the Garden of Guanabara Palace (Rio de Janeiro...
1,2013-07-24,Rio de Janeiro 2013,Brazil,Young people and students,111,Greeting to the young people from Italy at the conclusion of his v...
2,2013-07-24,Rio de Janeiro 2013,Brazil,"The poor, the sick, migrants",999,Visit to the St. Francis of Assisi of the Providence of God Hospit...
3,2013-07-24,Rio de Janeiro 2013,Brazil,Prayer and memory,168,Impromptu address of Pope Francis from the balcony of the Basilica...
4,2013-07-25,Rio de Janeiro 2013,Brazil,Young people and students,2155,Welcoming Ceremony by the young people on the waterfront of Copaca...


The table has one row per speech. The function `ac.charger()` adds three pieces of information.

- The country and the continent. The place written in the header is compared with a list of countries and cities. If it is missing, we take the country of the trip when there is only one, otherwise the country of the other speeches of the same day.
- The audience, deduced from the words of the title ("Meeting with Authorities", "Meeting with Young People"...).
- The length of the speech in words.

The next table sums up the corpus and checks that every speech has a country.

In [32]:
pd.Series({
    "Speeches": len(df),
    "Trips": df["voyage"].nunique(),
    "Countries": df["pays"].nunique(),
    "Speeches without a country": df["pays"].isna().sum(),
    "Speeches of fewer than 150 words": (df["n_mots"] < 150).sum(),
    "Words in total": df["n_mots"].sum(),
}, name="Number").to_frame()

,Number
Speeches,342
Trips,47
Countries,66
Speeches without a country,0
Speeches of fewer than 150 words,8
Words in total,543217


### Cleaning the text

Before counting themes, we look at the raw vocabulary. Words are counted in lowercase, so "God" and "god" are the same word. The most frequent words are "the", "of", "and" or "to", which say nothing about the content. We remove the English stop words (the list of scikit-learn), the words of fewer than three letters, and a few words of politeness that come back in every speech, such as "dear", "brothers", "sisters" or "thank". The full list is `MOTS_VIDES` in `outils/analyse_corpus.py`. About 40 % of the words remain.

In [33]:
words = [word for text in df["text"] for word in ac.mots(text)]
raw = pd.Series(Counter(words)).nlargest(20)
clean = pd.Series(Counter(w for w in words if len(w) > 2 and w not in ac.MOTS_VIDES)).nlargest(20)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Before cleaning", "After removing stop words"],
                    horizontal_spacing=0.15)
for column, (counts, color) in enumerate([(raw, g.GRIS), (clean, g.COULEURS[0])], start=1):
    fig.add_bar(x=counts.values[::-1], y=counts.index[::-1], orientation="h", row=1, col=column,
                showlegend=False, marker_color=color, hovertemplate="%{y}: %{x:,} times<extra></extra>")
fig.update_xaxes(showgrid=True)
g.titre(fig, "The 20 most frequent words of the 342 speeches", f"{len(words):,} words in all", hauteur=560)
fig.show()

After cleaning, "people", "God", "life", "Jesus" and "Church" come first. Beside the religious vocabulary, words about society appear: "world", "young", "peace", "hope", "human", "future", "country", "family", "work". "Peace" is the 10th word of the corpus, before "Lord" and "faith". These words guided the choice of the themes below.

### Where did he speak?

The trips themselves are described in part 1. Here the unit is the speech. Francis spoke in 66 countries. Brazil (15 speeches), the United States (14) and Poland (13) have the most speeches, because each hosted one long trip with many meetings: the World Youth Days in Rio and Kraków, and the visit to Washington, New York and Philadelphia. At the other end, 10 countries have only one or two speeches, which limits the comparisons by country below.

### Who does he speak to?

The audience is deduced from the title. For example "Meeting with the Authorities, Civil Society and the Diplomatic Corps" goes into "Authorities and diplomats".

In [34]:
audiences = df["public"].value_counts().sort_values()

fig = go.Figure(go.Bar(x=audiences.values, y=audiences.index, orientation="h", marker_color=g.COULEURS[0],
                       text=audiences.values, textposition="outside", cliponaxis=False,
                       hovertemplate="%{y}: %{x} speeches<extra></extra>"))
fig.update_xaxes(showgrid=True)
g.titre(fig, "Number of speeches by audience", hauteur=460)
fig.show()

Authorities come first: almost one speech in four (24 %) is given to political leaders, civil society or the diplomatic corps, and 41 of the 47 trips include such a meeting. The trips are therefore also state visits. Then come young people (16 %), ecumenical and interreligious meetings (15 %) and the poor, the sick and migrants (12 %). Bishops and clergy receive about one speech in five (19 %). Diplomacy and dialogue with other religions take as much room as the internal life of the Church.

### How long does he speak?

In [35]:
order = df.groupby("public")["n_mots"].median().sort_values().index.tolist()
medians = df.groupby("public")["n_mots"].median().reindex(order)

fig = px.strip(df, x="n_mots", y="public", hover_name="titre", custom_data=["voyage"],
               category_orders={"public": order[::-1]})
fig.update_traces(marker=dict(color=g.COULEURS[0], opacity=0.45, size=7), jitter=0.7,
                  hovertemplate="%{hovertext}<br>%{customdata[0]}, %{x} words<extra></extra>")
fig.add_scatter(x=medians.values, y=medians.index, mode="markers", showlegend=False,
                marker=dict(symbol="line-ns-open", size=26, color=g.ENCRE, line=dict(width=3, color=g.ENCRE)),
                hovertemplate="Median: %{x:.0f} words<extra></extra>")
fig.update_xaxes(title="Number of words", showgrid=True)
fig.update_yaxes(title=None)
g.titre(fig, "Length of the speeches by audience",
        "One point per speech, the black line marks the median. Hover over a point to read its title.", hauteur=540)
fig.show()

The median speech has about 1,400 words. Speeches to bishops, clergy and families are the longest, with medians above 2,000 words: they discuss the life of the Church in detail. Prayers and memorials (median of 650 words) and welcome and farewell speeches (860 words) are the shortest. The longest speech, about 5,000 words, is the address to the World Meeting of Popular Movements in Bolivia in 2015, a programme on land, housing and work for the poor. The shortest ones are a few lines, such as the message written in the golden book of the State House in Nairobi (18 words).

### Does he talk about the country he is in?

We now measure how much Francis adapts his speeches to the place and to the audience: the mention of the host country, the themes by country, by continent and by audience, the words specific to each continent and the similarity between speeches. We end with the link with the characteristics of the host country and the change over time. Oceania, with only 5 speeches, is left out of the comparisons by continent, as are the audiences with fewer than 10 speeches.

We first check whether the text names the country where the speech is given, for example "Kenya" in a speech in Nairobi.

In [36]:
df["names_host"] = ac.cite_le_pays_hote(df)

fig = make_subplots(rows=1, cols=2, subplot_titles=["By continent", "By audience"], horizontal_spacing=0.3)
for column, group in enumerate(["continent", "public"], start=1):
    shares = df.groupby(group)["names_host"].agg(["mean", "size"])
    shares = shares[shares["size"] >= 10].sort_values("mean")
    fig.add_bar(x=shares["mean"], y=shares.index, orientation="h", row=1, col=column, showlegend=False,
                width=0.5 if group == "continent" else 0.65,
                marker_color=[en.CONTINENT_COLORS.get(name, g.COULEURS[0]) for name in shares.index],
                text=[f"{p:.0%}" for p in shares["mean"]], textposition="outside", cliponaxis=False,
                customdata=shares["size"], hovertemplate="%{y}<br>%{x:.0%} of the %{customdata} speeches<extra></extra>")
fig.update_xaxes(range=[0, 1.1], tickformat=".0%", showgrid=True)
g.titre(fig, "Share of speeches that name the host country", "Groups of at least 10 speeches", hauteur=480)
fig.show()

Two speeches out of three (67 %) name the host country. The share is highest in Asia (85 %) and Africa (76 %), and lowest in Europe (56 %). By audience, the speeches to authorities almost always name the country (89 %), as do the welcome and farewell speeches (83 %), which address the whole nation. Speeches to young people (44 %) and the other meetings (30 %) name it least: they speak to a group rather than to a country. In Europe, part of the gap comes from speeches about Europe as a whole, as the specific words below show.

### Which themes come up most?

From the most frequent words, we define 15 themes by lists of English words. For example, the theme "Peace and war" gathers "peace", "war", "weapons", "conflict"... We count the mentions of each theme per 10,000 words. The measure is simple and easy to check, but it counts words and not ideas. The lists of words are in `outils/analyse_corpus.py`.

In [37]:
overall = ac.taux_lexique(df.assign(all_trips="all"), "all_trips", en.LEXICON).loc["all"].sort_values()

fig = go.Figure(go.Bar(x=overall.values, y=overall.index, orientation="h", marker_color=g.COULEURS[0],
                       text=overall.round(0).astype(int), textposition="outside", cliponaxis=False,
                       hovertemplate="%{y}: %{x:.1f} mentions per 10,000 words<extra></extra>"))
fig.update_xaxes(title="Mentions per 10,000 words", showgrid=True)
g.titre(fig, "How often each theme comes up in the 342 speeches", hauteur=520)
fig.show()

Peace and war comes far ahead, with 40 mentions per 10,000 words, about one every 250 words. Then come joy and hope (31), young people (22), the family (20) and the economy and work (19). Migrants (5.6), the elderly (6.4) and corruption and justice (6.8) are the rarest themes. They can still be central in a few countries, as the next section shows.

### Themes by country

This section follows the approach of Quentin. To compare countries, raw counts are misleading: in a country where the pope gave many long speeches, every theme gets more mentions. The example of migrants shows it, for the countries with at least 3 speeches.

In [38]:
counts = rb.mentions(df, en.LEXICON).groupby(df["pays"]).sum()
words_by_country = df.groupby("pays")["n_mots"].sum()
large_countries = df["pays"].value_counts().loc[lambda n: n >= 3].index

migrants = pd.DataFrame({
    "Number of mentions": counts.loc[large_countries, "Migrants and refugees"],
    "Mentions per 10,000 words": counts.loc[large_countries, "Migrants and refugees"]
                                 / words_by_country[large_countries] * 10_000,
})

fig = make_subplots(rows=1, cols=2, subplot_titles=list(migrants.columns), horizontal_spacing=0.22)
for column, measure in enumerate(migrants.columns, start=1):
    top = migrants[measure].nlargest(10).iloc[::-1]
    fig.add_bar(x=top.values, y=top.index, orientation="h", row=1, col=column, showlegend=False,
                marker_color=g.COULEURS[0], hovertemplate="%{y}: %{x:.1f}<extra></extra>")
fig.update_xaxes(showgrid=True)
g.titre(fig, "Migrants and refugees: the 10 countries where the theme comes up most",
        "Countries with at least 3 speeches", hauteur=460)
fig.show()

In number of mentions, Poland comes third, only because the pope gave 13 speeches there. Per 10,000 words, it falls to sixth place, and Turkey, with only 5 speeches, rises to second place. Morocco stays far ahead with both measures: during his visit in 2019, the pope met migrants in Rabat.

We then divide the rate of each theme in a country by its rate in the whole corpus. This index is above 1 when the pope talks more about the theme in this country than on average, and below 1 when he talks less about it. The heatmap shows the index of every theme in the 56 countries with at least 3 speeches, grouped by continent.

In [39]:
continent_of = df.groupby("pays")["continent"].first()
countries = sorted(large_countries, key=lambda c: (en.CONTINENT_ORDER.index(continent_of[c]), c))
rates = ac.taux_lexique(df, "pays", en.LEXICON).loc[countries]
index = rates / overall[rates.columns]
index = index[overall.sort_values(ascending=False).index]   # the most frequent themes first

fig = go.Figure(go.Heatmap(
    z=np.log2(index.clip(lower=1 / 8, upper=8)), x=list(index.columns),
    y=[f"{c} ({continent_of[c]})" for c in index.index], customdata=np.dstack([index, rates[index.columns]]),
    zmin=-3, zmax=3, xgap=1, ygap=1,
    colorscale=[[0, "#c0461b"], [0.25, "#f4a47c"], [0.5, "#efeee9"], [0.75, g.BLEUS[4]], [1, g.BLEUS[10]]],
    colorbar=dict(title="Index", tickvals=[-3, -2, -1, 0, 1, 2, 3],
                  ticktext=["1/8", "1/4", "1/2", "1", "2", "4", "8"], thickness=12, len=0.4),
    hovertemplate="%{y}, %{x}<br>Index %{customdata[0]:.2f}, %{customdata[1]:.1f} mentions per 10,000 words<extra></extra>"))
fig.update_xaxes(side="top", showline=False, tickangle=-45)
fig.update_yaxes(autorange="reversed", tickfont=dict(size=11))
g.titre(fig, "Index of each theme in each country (1 = average of all trips)",
        "Blue: more than on average, orange: less. Countries with at least 3 speeches, sorted by continent.",
        hauteur=1350, marge_haute=130)
fig.show()

In [40]:
# The 3 countries where each theme comes up most, with their index
top3 = index.apply(lambda column: ", ".join(f"{country} ({value:.1f})" for country, value in column.nlargest(3).items()))
top3.rename("countries where the theme comes up most (index)").to_frame()

,countries where the theme comes up most (index)
Peace and war,"Turkey (3.4), South Sudan (2.9), Sri Lanka (2.8)"
Joy and hope,"Cuba (2.5), Uganda (2.5), Turkey (1.8)"
Young people,"Timor-Leste (4.7), Papua New Guinea (2.8), Panama (2.7)"
Family,"Ireland (7.0), Philippines (5.0), United States (3.8)"
Economy and work,"Kenya (2.5), Bolivia (2.5), Albania (2.1)"
Missionary Church,"Thailand (4.1), Madagascar (2.8), Papua New Guinea (2.6)"
Memory and reconciliation,"Canada (4.1), Sri Lanka (3.7), Uganda (3.6)"
Poverty and exclusion,"Philippines (3.6), Kenya (2.9), Bolivia (2.8)"
Mercy and forgiveness,"Central African Republic (4.3), Poland (3.8), Sri Lanka (2.8)"
Interreligious dialogue,"Turkey (5.2), Bosnia and Herzegovina (4.6), Israel (4.5)"


Many results match the context of each visit. Migrants come up 17 times more than on average in Morocco, and also more in Turkey and Greece, two countries that receive many refugees. Interreligious dialogue stands out in Turkey, Bosnia and Herzegovina and Israel. The family stands out in Ireland, which hosted the World Meeting of Families in 2018, in the Philippines and in the United States. Memory and reconciliation stand out in Canada, where the pope apologised to the indigenous peoples in 2022, in Sri Lanka and in Uganda, where he honoured the martyrs of Namugongo. But most countries have only 3 to 8 speeches, and a single speech can put a country at the top: 38 cells of the heatmap are even at zero, because the theme never comes up in the country. The comparisons by continent, which gather more speeches, are more reliable.

### What does he talk about on each continent?

Grouping the countries by continent gives more speeches per group. The colour reads row by row: the darkest cell shows the continent where the theme comes up most, the lightest the one where it comes up least.

In [41]:
large_continents = [c for c in en.CONTINENT_ORDER if (df["continent"] == c).sum() >= 10]
rates = ac.taux_lexique(df, "continent", en.LEXICON).loc[large_continents].T
rates = rates.loc[rates.mean(axis=1).sort_values(ascending=False).index]

g.carte_chaleur(rates, "Mentions of each theme per 10,000 words, by continent",
                "Each row goes from the lightest (minimum) to the darkest (maximum)",
                unite="mentions per 10,000 words").show()

Peace and war is the most frequent theme everywhere, but its weight varies a lot: 93 mentions per 10,000 words in the Middle East against 19 in the Americas. Interreligious dialogue follows the same pattern (31 against 2). In the Americas, the pope talks more about the family (29) and young people (28), in line with the World Youth Days of Rio and Panama and the World Meeting of Families in Philadelphia. Africa stands out for the economy and work (27), corruption and justice (11) and migrants (11). Europe stands out for mercy and forgiveness (16). Other themes stay stable, such as joy and hope (25 to 33) and memory and reconciliation (12 to 17). The gaps match the context of the countries visited: wars and religious diversity in the Middle East, poverty and corruption in Africa, a mostly Catholic population in Latin America.

### And by audience?

Same measure, comparing the audiences with at least 20 speeches.

In [42]:
large_audiences = df["public"].value_counts().loc[lambda n: n >= 20].index
rates = ac.taux_lexique(df, "public", en.LEXICON).loc[large_audiences].T
rates = rates.loc[rates.mean(axis=1).sort_values(ascending=False).index]
rates.columns = [textwrap.fill(a, 16).replace("\n", "<br>") for a in rates.columns]

g.carte_chaleur(rates, "Mentions of each theme per 10,000 words, by audience",
                "Each row goes from the lightest (minimum) to the darkest (maximum)",
                unite="mentions per 10,000 words").show()

The vocabulary changes even more with the audience than with the continent. Peace and war dominates the speeches to authorities (71), to other religions (64) and the prayers and memorials (81), but it is rare before bishops and clergy (15). Some themes are almost kept for one audience: young people in speeches to young people (59), interreligious dialogue in ecumenical meetings (33), migrants in meetings with the poor and migrants (18), the missionary Church before bishops (39) and clergy (26), mercy in prayers (46). Ecology is mostly addressed to authorities (18), and human dignity to authorities (14) and to the poor (15). The pope chooses his themes according to his listeners: public questions for leaders, the life of the Church for its clergy.

### The words specific to each continent

For each continent, we look for the words that are more frequent than in the rest of the corpus. The score, the log-odds with a Dirichlet prior (Monroe, Colaresi and Quinn, 2008), takes into account the size of each group and limits the weight of rare words. Above 2, the gap is clear. Place names logically come first.

In [43]:
words_by_continent = ac.mots_caracteristiques(df, "continent", n=10)
continents = [c for c in en.CONTINENT_ORDER if c in words_by_continent]

fig = make_subplots(rows=1, cols=len(continents), subplot_titles=continents, shared_xaxes=True,
                    horizontal_spacing=0.08)
for column, continent in enumerate(continents, start=1):
    scores = words_by_continent[continent].iloc[::-1]
    fig.add_bar(x=scores.values, y=scores.index, orientation="h", row=1, col=column, showlegend=False,
                marker_color=en.CONTINENT_COLORS[continent],
                hovertemplate="%{y}: score %{x:.1f}<extra></extra>")
highest = max(scores.max() for scores in words_by_continent.values())
fig.update_xaxes(range=[0, highest * 1.05], showgrid=True)
g.titre(fig, "The 10 most specific words of each continent",
        "z-score of the log-odds, above 2 the gap is clear", hauteur=480)
fig.show()

Beyond place names, each continent has its own words. In Europe, "roots", "bridge" and "union" come from the speeches on the European project, and "sea" and "island" from the Mediterranean trips (Lesvos, Malta, Cyprus, Marseille, Corsica). In the Americas, "indigenous" and "peoples" point to the meetings with indigenous peoples, and "family" and "youth" to the World Youth Days and the World Meeting of Families. In Asia, "harmony", "respect" and "religions" reflect countries where Catholics are a small minority, and "nuclear" the visit to Hiroshima and Nagasaki. In the Middle East, "Abraham", "fraternity" and "religions" refer to the dialogue with Islam and Judaism. In Africa, "violence" and "corruption" echo the conflicts in the countries visited.

### Is a speech closer to its country or to its audience?

To compare whole speeches rather than a few themes, we turn each speech into a vector of word weights (TF-IDF). A word weighs more when it is frequent in the speech and rare in the rest of the corpus. The similarity of two speeches is the cosine of the angle between their vectors: 0 when they share no word, 1 when they use the same words in the same proportions. We keep only the words written in lowercase. Place names, names of people and words such as "Kenyan" disappear, so two speeches in the same country do not look alike only because they name the same places.

We then compare the pairs of speeches: same country and different audience, same audience and different country, or neither.

In [44]:
pairs = ac.similarite_des_paires(df)
pairs["kind"] = np.select(
    [pairs["meme_pays"] & pairs["meme_public"], pairs["meme_pays"], pairs["meme_public"]],
    ["Same country, same audience", "Same country, other audience", "Same audience, other country"],
    "Other country, other audience")
pairs.groupby("kind")["similarite"].agg(mean_similarity="mean", pairs="size").sort_values("mean_similarity").round(3)

,mean_similarity,pairs
kind,,
"Other country, other audience",0.109,49087
"Same country, other audience",0.126,842
"Same audience, other country",0.147,8248
"Same country, same audience",0.151,134


In [45]:
# Each pair counts for both of its speeches
both = pd.concat([pairs, pairs.rename(columns={"i": "j", "j": "i"})])
both["audience"] = df["public"].to_numpy()[both["i"]]
COMPARED = {"Same audience, other country": g.COULEURS[0], "Same country, other audience": g.COULEURS[1]}

large_audiences = df["public"].value_counts().loc[lambda n: n >= 20].index
means = both[both["audience"].isin(large_audiences)].groupby(["audience", "kind"])["similarite"].mean().unstack()
means = means.sort_values("Same audience, other country")
baseline = pairs.loc[pairs["kind"] == "Other country, other audience", "similarite"].mean()

fig = go.Figure()
for audience, row in means.iterrows():
    fig.add_scatter(x=[row["Same country, other audience"], row["Same audience, other country"]], y=[audience] * 2,
                    mode="lines", line=dict(color=g.GRILLE, width=3), showlegend=False, hoverinfo="skip")
for kind, color in COMPARED.items():
    fig.add_scatter(x=means[kind], y=means.index, mode="markers", name=kind,
                    marker=dict(color=color, size=12, line=dict(color=g.FOND, width=1.5)),
                    hovertemplate="%{y}<br>" + kind + ": %{x:.3f}<extra></extra>")
fig.add_vline(x=baseline, line=dict(color=g.GRIS, width=1, dash="dot"),
              annotation_text="other country, other audience", annotation_position="bottom right",
              annotation_font=dict(size=12, color=g.GRIS))
fig.update_xaxes(title="Mean cosine similarity", showgrid=True)
g.titre(fig, "Is a speech closer to the same audience abroad or to its own country?",
        "Mean similarity of the speeches of each audience with two groups of other speeches", hauteur=480)
fig.show()

Two speeches given to the same audience in different countries are more alike (0.147 on average) than two speeches given in the same country to different audiences (0.126). Both are more alike than two speeches with nothing in common (0.109), so the place and the audience both count, but the audience counts more. 59 % of the speeches are closer to the speeches given to the same audience abroad than to the other speeches of their own country. The gap is largest for the institutional audiences: authorities, bishops and clergy receive speeches that look alike from one country to the next. On the contrary, speeches to the poor, the sick and migrants, and prayers and memorials, are closer to the other speeches of the same country. They are the most tied to the place.

### Does the host country shape the themes?

We now link the speeches to the data of part 1. For each host country, we take the share of Catholics and the GDP per capita at the time of the visit, and compare them with the rate of four themes in the speeches given there. We state four expectations before looking at the results. The pope should talk more about interreligious dialogue where Catholics are few, and more about the family where they are many. He should talk more about poverty, and about corruption and justice, in poorer countries. The rho is the Spearman rank correlation: -1 when the theme always decreases with the variable, 1 when it always increases, 0 when there is no link. We keep the countries with at least 3 speeches.

In [46]:
# Religion and wealth of each host country, from part 1 (closest year to the visit)
host = visits[visits["pope"] == "Francis"].groupby("iso3")[["catholics", "gdp_per_capita"]].mean()
host = (df.groupby("iso").agg(country=("pays", "first"), speeches=("titre", "size"))
          .join(host).join(ac.taux_lexique(df, "iso", en.LEXICON)))
host = host[host["speeches"] >= 3].dropna(subset=["catholics", "gdp_per_capita"])

EXPECTATIONS = [("Interreligious dialogue", "catholics"), ("Family", "catholics"),
                ("Poverty and exclusion", "gdp_per_capita"), ("Corruption and justice", "gdp_per_capita")]
AXIS_TITLES = {"catholics": "Share of Catholics (%)", "gdp_per_capita": "GDP per capita (2015 dollars, log scale)"}

titles = []
for theme, variable in EXPECTATIONS:
    rho, p_value = spearmanr(host[variable], host[theme])
    titles.append(f"{theme} (rho = {rho:.2f}, " + ("p < 0.01)" if p_value < 0.01 else f"p = {p_value:.2f})"))

fig = make_subplots(rows=2, cols=2, subplot_titles=titles, vertical_spacing=0.2, horizontal_spacing=0.1)
for k, (theme, variable) in enumerate(EXPECTATIONS):
    row, column = k // 2 + 1, k % 2 + 1
    fig.add_scatter(x=host[variable], y=host[theme], mode="markers", row=row, col=column, showlegend=False,
                    text=host["country"], customdata=host["speeches"],
                    marker=dict(size=5 + host["speeches"], color=g.COULEURS[0], opacity=0.6,
                                line=dict(color=g.FOND, width=1)),
                    hovertemplate="%{text}, %{customdata} speeches<br>%{y:.1f} mentions per 10,000 words<extra></extra>")
    if variable == "gdp_per_capita":
        fig.update_xaxes(type="log", tickvals=[1000, 3000, 10000, 30000, 100000],
                         ticktext=["1k", "3k", "10k", "30k", "100k"], row=row, col=column)
    fig.update_xaxes(title_text=AXIS_TITLES[variable], row=row, col=column)
fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True, rangemode="tozero", title_text="Mentions per 10,000 words", col=1)
fig.update_annotations(font_size=13)
g.titre(fig, "Four themes and the characteristics of the host country",
        f"One point per country with at least 3 speeches ({len(host)} countries), the size gives the number of speeches",
        hauteur=720)
fig.show()

The first expectation is clearly confirmed: the fewer Catholics in a country, the more Francis talks about interreligious dialogue (rho = -0.64). The theme comes up most in Turkey, Bosnia and Herzegovina, Israel, Morocco and Indonesia, where Muslims or Jews are the majority or a large part of the population. But a small Catholic minority is not enough. In South Korea, Japan, Thailand or Estonia, where Catholics are few and Muslims too, the theme is rare. The dialogue that Francis puts forward is mainly a dialogue with Islam and Judaism.

The other expectations are not confirmed. The family theme is not linked to the share of Catholics (rho = 0.20, p = 0.15). Its peaks come from specific events, such as the World Meetings of Families in Dublin and Philadelphia. Poverty is not more frequent in poorer countries (rho = -0.11), and corruption and justice only slightly (rho = -0.24, p = 0.07). With 55 countries and speeches to very different audiences, these correlations stay fragile.

### Does the discourse change over time?

We follow four themes year by year. The grey line gives the average over all trips. Some years have only one or two trips, so the changes there are fragile.

In [47]:
themes = ["Peace and war", "Migrants and refugees", "Ecology and creation", "Interreligious dialogue"]
by_year = ac.taux_lexique(df, "annee", en.LEXICON).reindex(range(2013, 2025))
average = ac.taux_lexique(df.assign(all_trips="all"), "all_trips", en.LEXICON).loc["all"]

fig = make_subplots(rows=2, cols=2, subplot_titles=themes, shared_xaxes=True,
                    vertical_spacing=0.16, horizontal_spacing=0.08)
for i, (theme, color) in enumerate(zip(themes, g.COULEURS)):
    row, column = i // 2 + 1, i % 2 + 1
    fig.add_scatter(x=by_year.index, y=by_year[theme], mode="lines+markers", row=row, col=column,
                    line=dict(color=color, width=2), marker=dict(size=8, line=dict(color=g.FOND, width=1.5)),
                    showlegend=False, hovertemplate="%{x}: %{y:.1f} mentions per 10,000 words<extra></extra>")
    fig.add_hline(y=average[theme], line=dict(color=g.GRIS, width=1), layer="below", row=row, col=column)
fig.update_xaxes(dtick=2)
fig.update_yaxes(rangemode="tozero", showgrid=True)
g.titre(fig, "Mentions per 10,000 words, year by year",
        "The grey line gives the average over all trips", hauteur=580)
fig.show()

The peaks follow specific trips. Peace and war comes back strongly in 2014 (Holy Land, Strasbourg, Turkey), 2017 and 2022, the year of the war in Ukraine and of the trips to Kazakhstan and Bahrain. Migrants peak in 2016, with the visit to the refugee camp of Lesvos, and in 2019, with the trip to Morocco. Ecology peaks in 2015, the year of the encyclical Laudato si', and again in 2022. Interreligious dialogue rises in 2014, 2021 (Iraq) and 2022 (Kazakhstan, Bahrain). For these four themes there is no steady rise or fall over the period.

## Part 3. Are the differences robust?

This part follows the method proposed by Quentin. Part 2 compares rates of mentions between groups of speeches. Two doubts remain.

1. Chance. A continent has between 41 and 105 speeches, and one long speech centred on a single theme can be enough to move its rate. We answer this doubt with a bootstrap.
2. Confounding. The continent and the audience are linked. If peace comes up more in Africa, is it because of Africa, or because the pope meets more authorities there? We answer this doubt with a regression, which compares the continents for the same audience and the same year.

The code is in `outils/robustness.py`.

### Bootstrap: is a gap larger than chance?

The index of a theme on a continent is its rate of mentions on this continent divided by its rate in the whole corpus. An index of 2 means that the theme comes up twice as often as on average. To measure how fragile it is, we replay the trips 1,000 times. Each time, we draw at random, with replacement, as many speeches as the continent has (some come twice, others disappear) and compute the index again. The middle 95 % of the 1,000 values form the confidence interval.

- If the whole interval is above 1, the pope really talks more about the theme on this continent.
- If it is below 1, he really talks less about it.
- If it contains 1, we cannot conclude.

In [48]:
boot = rb.bootstrap(df, "continent", en.LEXICON, draws=1000, minimum=10)
continents = [c for c in en.CONTINENT_ORDER if c in set(boot["continent"])]
theme_order = list(ac.taux_lexique(df.assign(all_trips="all"), "all_trips", en.LEXICON).loc["all"].sort_values().index)

VERDICTS = {"more": ("More than average", g.COULEURS[0]),
            "less": ("Less than average", g.COULEURS[1]),
            "no clear gap": ("Interval contains 1", g.GRIS)}

fig = make_subplots(rows=1, cols=len(continents), subplot_titles=continents, shared_yaxes=True,
                    horizontal_spacing=0.015)
in_legend = set()
for column, continent in enumerate(continents, start=1):
    part = boot[boot["continent"] == continent].set_index("theme").loc[theme_order]
    for verdict, (label, color) in VERDICTS.items():
        sub = part[part["verdict"] == verdict]
        # One segment per theme, from the low end to the high end of the interval
        fig.add_scatter(x=np.column_stack([sub["low"], sub["high"], np.full(len(sub), np.nan)]).ravel(),
                        y=np.repeat(sub.index, 3), mode="lines", line=dict(color=color, width=2),
                        hoverinfo="skip", showlegend=False, row=1, col=column)
        fig.add_scatter(x=sub["index"], y=sub.index, mode="markers", name=label, legendgroup=verdict,
                        showlegend=verdict not in in_legend and len(sub) > 0, row=1, col=column,
                        marker=dict(color=color, size=9, line=dict(color=g.FOND, width=1.5)),
                        customdata=sub[["low", "high"]],
                        hovertemplate="%{y}<br>Index %{x:.2f}, interval from %{customdata[0]:.2f}"
                                      " to %{customdata[1]:.2f}<extra></extra>")
        if len(sub) > 0:
            in_legend.add(verdict)
fig.add_vline(x=1, line=dict(color=g.ENCRE_2, width=1, dash="dot"), row="all", col="all")
fig.update_xaxes(type="log", range=[np.log10(0.08), np.log10(6)], showgrid=True, tickangle=0,
                 tickvals=[0.25, 1, 4], ticktext=["1/4", "1", "4"])
fig.update_layout(legend_y=1.07)
g.titre(fig, "Index of each theme by continent, with its 95 % bootstrap interval",
        "1 is the average of all trips. Logarithmic scale: 2 and 1/2 are at the same distance from 1.",
        hauteur=660, marge_haute=30)
fig.show()

In [49]:
# Number of themes with a clear gap on each continent
pd.crosstab(boot["continent"], boot["verdict"]).loc[continents]

verdict,less,more,no clear gap
continent,,,
Europe,2,0,13
Americas,3,0,12
Asia,1,0,14
Middle East,5,2,8
Africa,1,1,13


Most gaps of part 2 do not survive the bootstrap: on each continent, the interval contains 1 for 8 to 14 themes out of 15. The Middle East is the most distinctive continent, with 7 clear gaps: more peace and more interreligious dialogue, and less young people, family, economy, missionary Church and poverty. In the Americas, the pope really talks less about peace, interreligious dialogue and migrants. In Africa, he talks more about the economy and work and less about the family. Europe and Asia stay close to the average. Some striking gaps of the heatmap rest on a few speeches only. Migrants in Africa have an index of 2, but its interval goes from 0.4 to 5.

### Regression: is it the place or the audience?

For each theme, we model the number of mentions in each speech. The explanatory variables are the continent, the audience and the year. The length of the speech enters as an offset, so the model explains a number of mentions per word, like the rates of part 2.

We use a negative binomial model rather than a Poisson model, because the counts are very dispersed: many speeches never mention a theme and a few mention it dozens of times. A Poisson model assumes that the variance of the counts equals their mean. The table shows that here it is between 3.6 and 19.5 times larger.

In [50]:
counts = rb.mentions(df, en.LEXICON)
pd.DataFrame({"mean": counts.mean(), "variance": counts.var(),
              "variance / mean": counts.var() / counts.mean()}).sort_values("variance / mean").round(1)

,mean,variance,variance / mean
Corruption and justice,1.1,3.8,3.6
Human dignity,1.1,4.5,4.0
Memory and reconciliation,2.1,9.4,4.5
Interreligious dialogue,1.6,7.9,4.8
The elderly,1.0,4.9,4.8
Joy and hope,4.8,26.8,5.5
Economy and work,3.1,17.3,5.6
Poverty and exclusion,2.1,11.8,5.7
Ecology and creation,1.3,7.6,5.9
Mercy and forgiveness,1.9,13.0,6.8


The reference continent is Europe and the reference audience is the authorities and diplomats, the largest group. Oceania (5 speeches) is left out, and the audiences with fewer than 10 speeches join the other meetings. The year enters as a linear trend.

Each coefficient is shown as a rate ratio. A ratio of 2 for Africa means twice as many mentions per word as in Europe, for the same audience and the same year. A ratio of 0.5 means half as many. With 15 themes and 13 variables, we run 195 tests, and some would come out significant by chance only. We therefore adjust the p-values with the Benjamini-Hochberg method and keep the effects with an adjusted p-value below 5 %.

In [51]:
data = df[df["continent"] != "Oceania"].copy()
small = data["public"].map(data["public"].value_counts()) < 10
data.loc[small, "public"] = "Other meetings"

reg = rb.regressions(data, en.LEXICON, reference_continent="Europe", reference_audience="Authorities and diplomats")
reg["significant"] = reg["p_adjusted"] < 0.05
reg.groupby("kind")["significant"].agg(significant="sum", tests="size")

,significant,tests
kind,,
Audience,56,120
Continent,15,60
Year,4,15


In [52]:
# The trend is shown over ten years rather than one, to make it readable
shown = reg.copy()
trend = shown["kind"] == "Year"
shown.loc[trend, ["rate_ratio", "low", "high"]] **= 10
shown.loc[trend, "variable"] = "Ten years later"

panels = {
    "Continent": [c for c in en.CONTINENT_ORDER if c in set(data["continent"]) - {"Europe"}],
    "Audience": [a for a in data["public"].value_counts().index if a != "Authorities and diplomats"],
    "Year": ["Ten years later"],
}
titles = ["Continent, vs Europe", "Audience, vs authorities", "Trend"]
rows = theme_order[::-1]   # the most frequent themes at the top
SHORT = {"Young people and students": "Young people", "Ecumenical and interreligious": "Ecumenical",
         "The poor, the sick, migrants": "Poor, sick, migrants", "Clergy and religious": "Clergy",
         "Prayer and memory": "Prayer, memory", "Other meetings": "Other", "Welcome and farewell": "Welcome, farewell"}

fig = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=titles, horizontal_spacing=0.015,
                    column_widths=[len(columns) for columns in panels.values()])
for column, (kind, columns) in enumerate(panels.items(), start=1):
    table = shown[shown["kind"] == kind]
    pivot = {name: table.pivot(index="theme", columns="variable", values=name).loc[rows, columns]
             for name in ["rate_ratio", "low", "high", "p_adjusted", "significant"]}
    significant = pivot["significant"].astype(bool)
    # Colour only the significant effects, on a log scale capped at 4 and 1/4.
    # A ratio that cannot be estimated stays empty.
    ratio = pivot["rate_ratio"]
    color = np.log2(ratio).clip(-2, 2).where(significant | ratio.isna(), 0)
    text = ratio.map(lambda r: f"{r:.2g}").where(significant, "")
    fig.add_heatmap(z=color, x=[SHORT.get(c, c) for c in columns], y=rows,
                    text=text, texttemplate="%{text}", row=1, col=column,
                    customdata=np.dstack([pivot[name].to_numpy(dtype=float) for name in ["rate_ratio", "low", "high", "p_adjusted"]]),
                    zmin=-2, zmax=2, xgap=2, ygap=2, showscale=column == 3,
                    colorscale=[[0, "#c0461b"], [0.25, "#f4a47c"], [0.5, "#efeee9"], [0.75, g.BLEUS[4]], [1, g.BLEUS[10]]],
                    colorbar=dict(title="Rate ratio", tickvals=[-2, -1, 0, 1, 2], ticktext=["1/4", "1/2", "1", "2", "4"],
                                  thickness=12, len=0.6),
                    hovertemplate="%{y}, %{x}<br>Rate ratio %{customdata[0]:.2f}, 95 % interval from %{customdata[1]:.2f}"
                                  " to %{customdata[2]:.2f}<br>Adjusted p-value %{customdata[3]:.3f}<extra></extra>")
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(showline=False, tickangle=-45)
fig.update_layout(margin_b=120)
g.titre(fig, "Effect of the continent, the audience and the year on each theme",
        "Only the effects with an adjusted p-value below 5 % are coloured. Hover over a cell for the details.",
        hauteur=800)
fig.show()

One cell has no value: the migration theme never comes up in welcome and farewell speeches, so its ratio cannot be estimated.

Once the audience is taken into account, part of the continent effects remain. Compared with Europe, the pope talks about twice as much about peace in Africa (2.1) and in the Middle East (2.2), and about half as much in the Americas (0.57). Interreligious dialogue is four times rarer in the Americas (0.25) and more frequent in the Middle East (1.8). Corruption and justice come up three times more in Africa (2.9) and about twice more in Asia and the Americas. Migrants are eight times rarer in the Americas (0.12).

The audience matters more than the place: 56 audience effects out of 120 are significant, against 15 continent effects out of 60. Mercy is 2.5 to 20 times more frequent before any audience than before authorities. Public questions (peace, ecology, human dignity, corruption, economy) are up to eight times rarer before bishops and clergy.

The bootstrap and the regression agree on the main points: more peace and interreligious dialogue in the Middle East, less peace, interreligious dialogue and migrants in the Americas, more economy in Africa. Other gaps disappear: less family in Africa, or less economy and poverty in the Middle East, came from the type of meetings held there. Finally, four themes lose weight over time. In ten years, the family theme is divided by three (0.34), and poverty (0.53), human dignity (0.46) and the economy (0.63) by about two.

## Limits

- The speeches are the English versions published by the Vatican. Many of them were given in Spanish or Italian, so we often work on a translation.
- The country and the audience of each speech are deduced automatically from the header and the title. A few errors remain possible, for example "Georgia" can be the American state.
- The themes rely on lists of words that we chose. A word can have another meaning, and an idea can be expressed without these words.
- The groups have unequal sizes. Oceania has 5 speeches and some years only one trip. At the country level, most countries have fewer than 8 speeches, too few for a reliable bootstrap, which is why part 3 stays at the level of continents.
- RCS-Dem stops in 2015, so the religious composition of the countries visited by Francis after 2015 is the one of 2015. The World Bank income groups are the current ones, not those of the year of the visit.
- The regression shows associations, not causes. The programme of a trip, and so the audiences met, depends on the country visited.

- Speeches of the same trip share the context of the moment. In the bootstrap and the regression they are treated as independent, which makes the intervals somewhat too narrow.
- The similarity between speeches also depends on time: two speeches of the same trip are close in time, which can make them alike for reasons that have nothing to do with the country.

## Conclusion

Francis travelled as much as John Paul II, about four trips a year. Like his predecessors, he went mostly to countries more Catholic than the rest of the world, but his trips are the least oriented towards Catholic countries of the three: a third of the countries he visited have almost no Catholics, as many as among the countries he did not visit. Two thirds of his visits were in countries where Catholicism did better than the world average, which points to an "offensive" rather than a defensive diplomacy. On the other hand, he did not favour poor countries: half of his visits went to countries richer than the world average, and he visited 24 % of the low income countries, against 56 % for John Paul II.

Francis does adapt his speeches, but less to the country than to the audience. He names the host country in two speeches out of three, and some themes clearly follow the place: peace and interreligious dialogue in the Middle East, the economy and corruption in Africa, less peace and less dialogue in the Americas. These gaps resist the bootstrap and the regression. But the audience shapes the content much more. On average, a speech to bishops looks more like the speeches to bishops in other countries than like the other speeches of its own trip. Leaders hear about peace, ecology and human dignity, the clergy about mission and mercy, young people about youth and hope. The pope keeps a common message for each type of listener and adds to it the topics of the place he visits.